# DASHBOARD PYTHON AVANCEE


En utilisant le jeu de données `supermarket_sales.csv`, vous devez réaliser un tableau de bord interactif 
avec `Dash` qui permet d'analyser les ventes d'un supermarché.

Vous choisirez deux (2) des indicateurs et trois (3) des graphiques présentés ci-dessous dans 
votre tableau de bord. Chaque indicateur et graphique doit être interactif en fonction du **sexe** et de la **ville**. 

Indicateurs par sexe et par ville :

- Montant total des achats : Somme du montant total des achats (Total)
- Nombre total d'achats : Nombre total d'achats (Invoice ID)
- Évaluation moyenne : Moyenne des évaluations (Rating)

Graphiques :

- Histogramme donnant la répartition des montants totaux des achats par sexe et par ville.
- Diagramme en barres du nombre total d'achats par sexe et par ville.
- Diagramme circulaire montrant la répartition de la catégorie de produit (Product line) par sexe et par ville.
- Evolution du montant total des achats par semaine par ville.

## Importation des librairie 

In [87]:
import pandas as pd 
import numpy as np 

# graphique 
import plotly.graph_objects as go
import plotly.express as px

# Dash 
import dash 
from dash import html, dcc 
import dash_bootstrap_components as dbc 
from dash import dash_table

## Importation, visualisation & compréhension des data 

In [31]:
data = pd.read_csv("data/supermarket_sales.csv")
display(data.head())

,Invoice ID,Branch,City,Customer type,Gender,Product line,Unit price,Quantity,Tax 5%,Total,Date,Time,Payment,cogs,gross margin percentage,gross income,Rating
0,750-67-8428,A,Yangon,Member,Female,Health and beauty,74.69,7,26.1415,548.9715,1/5/2019,13:08,Ewallet,522.83,4.761905,26.1415,9.1
1,226-31-3081,C,Naypyitaw,Normal,Female,Electronic accessories,15.28,5,3.8200,80.2200,3/8/2019,10:29,Cash,76.40,4.761905,3.8200,9.6
2,631-41-3108,A,Yangon,Normal,Male,Home and lifestyle,46.33,7,16.2155,340.5255,3/3/2019,13:23,Credit card,324.31,4.761905,16.2155,7.4
3,123-19-1176,A,Yangon,Member,Male,Health and beauty,58.22,8,23.2880,489.0480,1/27/2019,20:33,Ewallet,465.76,4.761905,23.2880,8.4
4,373-73-7910,A,Yangon,Normal,Male,Sports and travel,86.31,7,30.2085,634.3785,2/8/2019,10:37,Ewallet,604.17,4.761905,30.2085,5.3


In [32]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Invoice ID               1000 non-null   object 
 1   Branch                   1000 non-null   object 
 2   City                     1000 non-null   object 
 3   Customer type            1000 non-null   object 
 4   Gender                   1000 non-null   object 
 5   Product line             1000 non-null   object 
 6   Unit price               1000 non-null   float64
 7   Quantity                 1000 non-null   int64  
 8   Tax 5%                   1000 non-null   float64
 9   Total                    1000 non-null   float64
 10  Date                     1000 non-null   object 
 11  Time                     1000 non-null   object 
 12  Payment                  1000 non-null   object 
 13  cogs                     1000 non-null   float64
 14  gross margin percentage  

In [33]:
# Transformation du type en format data 
data["Date"] = pd.to_datetime(data["Date"], errors='coerce')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Invoice ID               1000 non-null   object        
 1   Branch                   1000 non-null   object        
 2   City                     1000 non-null   object        
 3   Customer type            1000 non-null   object        
 4   Gender                   1000 non-null   object        
 5   Product line             1000 non-null   object        
 6   Unit price               1000 non-null   float64       
 7   Quantity                 1000 non-null   int64         
 8   Tax 5%                   1000 non-null   float64       
 9   Total                    1000 non-null   float64       
 10  Date                     1000 non-null   datetime64[ns]
 11  Time                     1000 non-null   object        
 12  Payment                  1000 non-n

# Description des variables 

**Identifiants et Localisation**  
- Invoice ID : Le numéro de facture unique pour chaque transaction.
- Branch : La succursale du magasin (ici identifiée par des lettres : A, B, C).
- City : La ville où se situe le magasin.  
  
**Profil**    
-  ClientCustomer type : Le segment du client. "Member" pour ceux qui ont une carte de fidélité, "Normal" pour les autres.
- Gender : Le genre du client (Male/Female).Détails du ProduitProduct line : La catégorie du produit acheté (Santé et beauté, Accessoires électroniques, etc.).
- Unit price : Le prix d'une seule unité du produit.Quantity : Le nombre d'unités achetées.    


**Calculs Financiers**
- Tax 5% : Le montant de la taxe sur la valeur ajoutée (TVA) calculé sur l'achat ($5\%$).
- Total : Le prix final payé par le client (Prix unitaire $\times$ Quantité + Taxe).cogs : Signifie 
- Cost of Goods Sold (Coût des marchandises vendues). C'est ce que le produit a coûté au magasin pour l'acheter ou le fabriquer.
- gross margin percentage : Le pourcentage de marge brute. C'est le ratio de profitabilité.
- gross income : Le bénéfice brut généré par la vente ($Total - cogs - Taxe$ ou simplement la marge en valeur monétaire).  

**Logistique et Satisfaction**
- Date / Time : Le moment précis de la transaction.
- Payment : Le mode de paiement utilisé (Ewallet, Cash, Credit Card).
- Rating : La note de satisfaction donnée par le client sur une échelle de 1 à 10.

## Préparation des graphiques 

### Graphique: diagramme répartition des catégories de produits par sexe et par ville

In [34]:
# fonction qui calcul la répartition des catégories de produits par sexes
def repartition_cat_produits(data): 
    """
    La fonction permet de calculer le nombre de commande pour chaque catégorie de produit en fonction 
    du sexe et de la ville. 
    Elle prend en entrée un dataframe et renvoie en sorti un autre dataframe contenant le genre,
    le ville, la catégorie de produit et le nombre de commande. 
    """
    
    data_cat_prod = data.groupby(["Gender", "City", 'Product line']).size().reset_index()
    data_cat_prod = data_cat_prod.rename(columns = {0: "Nombre_commande"}) 
    
    return data_cat_prod 


In [35]:
repartition_cat_produits(data).head()

,Gender,City,Product line,Nombre_commande
0,Female,Mandalay,Electronic accessories,28
1,Female,Mandalay,Fashion accessories,33
2,Female,Mandalay,Food and beverages,29
3,Female,Mandalay,Health and beauty,20
4,Female,Mandalay,Home and lifestyle,22


In [36]:
# Graphique diagramme circulaire 
data_rep = repartition_cat_produits(data)
data_rep.columns

df_filtre = data_rep[(data_rep["City"] == "Mandalay") & (data_rep["Gender"] == "Female")]

graphe_cat_prod = px.pie(
    data_frame= df_filtre,
    values = "Nombre_commande",
    names = "Product line",
    title= "Répartition"
)

graphe_cat_prod.show()

### Graphique: diagramme en bar nombre total d'achat par sexe et par ville 

In [37]:
def total_achat(data): 
    '''
    La fonction prend en entrée un dataframe puis renvoie un autre dataframe en sortie contenant 
    le nombre d'achat par sexe et par ville 
    '''

    df = data.groupby(['Gender', 'City']).size().reset_index()
    df = df.rename(columns = {0: "Nombre_total_achat"}) 

    return df 

In [38]:
total_achat(data)

,Gender,City,Nombre_total_achat
0,Female,Mandalay,162
1,Female,Naypyitaw,178
2,Female,Yangon,161
3,Male,Mandalay,170
4,Male,Naypyitaw,150
5,Male,Yangon,179


In [39]:
df = total_achat(data)
df_f = df[(df["City"] == "Mandalay")]

diagramme_bar = px.bar(
    data_frame= df_f,
    x = 'City',
    y = 'Nombre_total_achat',
    color="Gender",
    barmode='group',
    title = "Répartition du nombre total d'achat par sexe et par ville "
)




In [40]:
diagramme_bar.show()


### Graphique: Histogramme montant totaux par ville et par genre

In [41]:
# Fonction permettant de calculer le montant des achats par ville et par colonne 
def montant_achat(data): 
    """
    La fonction prend en entrée un dataframe et ressort également un dataframe dans lequel se 
    trouve le montant total des achats par sexe et par ville en euro(€)
    """
    df = data.groupby(['City', 'Gender']).sum('Total').reset_index()
    df = df[['City', 'Gender', 'Total']]

    return df 
  

In [42]:
montant_achat(data)

,City,Gender,Total
0,Mandalay,Female,52928.2950
1,Mandalay,Male,53269.3770
2,Naypyitaw,Female,61685.4630
3,Naypyitaw,Male,48883.2435
4,Yangon,Female,53269.1670
5,Yangon,Male,52931.2035


In [43]:
dff = montant_achat(data)
dff_f = dff[(dff['City']== "Mandalay")]

histogramme_montant_total = px.histogram(
    data_frame= dff_f,
    y = 'Total',
    x = 'City',
    color = 'Gender',
    barmode= 'group',
    title = 'Montant total des achats par ville et par sexe '
)

In [44]:
histogramme_montant_total.show()

In [45]:
# sans fonction au préalable 
histogramme = px.histogram(
    data_frame= data,
    y = 'Total',
    x = 'City',
    color = 'Gender',
    barmode= 'group',
    title = 'Montant total des achats par ville et par sexe '
)

In [46]:
histogramme.show()

## préparation indicateur 
### Indicateur nombre total d'achat 

In [ ]:
def total_achat(data): 
    '''
    La fonction prend en entrée un dataframe puis renvoie un autre dataframe en sortie contenant 
    le nombre d'achat par sexe et par ville 
    '''

    df = data.groupby(['Gender', 'City']).size().reset_index()
    df = df.rename(columns = {0: "Nombre_total_achat"}) 

    return df 

In [80]:
df1 = total_achat(data)
df1['Gender'] = df1['Gender'].replace("Female", "Femme")
df1['Gender'] = df1['Gender'].replace("Male", "Homme")
df1

,Gender,City,Nombre_total_achat
0,Femme,Mandalay,162
1,Femme,Naypyitaw,178
2,Femme,Yangon,161
3,Homme,Mandalay,170
4,Homme,Naypyitaw,150
5,Homme,Yangon,179


In [81]:
# Filtre
nb_choisi = df1[(df1["City"] == "Mandalay") & (df1["Gender"] == "Femme")].values[0]
display(nb_choisi)
nb_autre = df1[(df1["City"] == "Mandalay") & (df1["Gender"] == "Homme")].values[0]
display(nb_autre)

array(['Femme', 'Mandalay', 162], dtype=object)

array(['Homme', 'Mandalay', 170], dtype=object)

In [86]:

indicateur_nb = go.Figure()

indicateur_nb.add_trace(
    go.Indicator(value = nb_choisi[2], 
        mode = "number+delta"
    )
)

indicateur_nb.update_layout( 
    
    template = {'data' : {'indicator': [{ 
    	'title': {'text': f"Nombre total d'achats des {nb_choisi[1]} à {nb_choisi[0]}"}, 
        'mode' : "number+delta+gauge",
        'delta' : {'reference':nb_autre[2]
                    }}]
        }
     }
)

indicateur_nb.show()

### Indicateur montant total d'achat 

In [58]:
# Fonction permettant de calculer le montant des achats par ville et par colonne 
def montant_achat(data): 
    """
    La fonction prend en entrée un dataframe et ressort également un dataframe dans lequel se 
    trouve le montant total des achats par sexe et par ville en euro(€)
    """
    df = data.groupby(['City', 'Gender']).sum('Total').reset_index()
    df = df[['City', 'Gender', 'Total']]

    return df 

In [75]:
df2 = montant_achat(data)
df2['Gender'] = df2['Gender'].replace("Female", "Femme")
df2['Gender'] = df2['Gender'].replace("Male", "Homme")
df2

,City,Gender,Total
0,Mandalay,Femme,52928.2950
1,Mandalay,Homme,53269.3770
2,Naypyitaw,Femme,61685.4630
3,Naypyitaw,Homme,48883.2435
4,Yangon,Femme,53269.1670
5,Yangon,Homme,52931.2035


In [78]:
# Filtre
montant_choisi = df2[(df2["City"] == "Mandalay") & (df2["Gender"] == "Femme")].values[0]
display(montant_choisi)
montant_autre = df2[(df2["City"] == "Mandalay") & (df2["Gender"] == "Homme")].values[0]
display(montant_autre)

array(['Mandalay', 'Femme', 52928.295], dtype=object)

array(['Mandalay', 'Homme', 53269.377], dtype=object)

In [79]:


indicateur_montant = go.Figure()

indicateur_montant.add_trace(
    go.Indicator(value = montant_choisi[2], 
        mode = "number+delta"
    )
)

indicateur_montant.update_layout( 
    
    template = {'data' : {'indicator': [{ 
    	'title': {'text': f'Montant total des achats des {montant_choisi[1]} à {montant_choisi[0]}'}, 
        'mode' : "number+delta+gauge",
        'delta' : {'reference':montant_autre[2] }}]
        }
     }
)

indicateur_montant.show()

## Création de la maquette

In [100]:
app = dash.Dash(external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container(fluid=True,
    children=[
        dbc.Row(children='a',style={'height':'10vh', 'background-color':'skyblue','border':'solid red'}),
        dbc.Row(children=[
            dbc.Col(children='b',
                    style= {"width": "50%", "border":"solid red"}),
            dbc.Col(children='c',
                    style= {"width": "50%", "border":"solid red"})
        ], style= {"height": "15vh", "border":"solid red"}),
        dbc.Row(children=[
            dbc.Col(children='d',
                    style= {"width": "50%", "border":"solid red"}),
            dbc.Col(children='e',
                    style= {"width": "50%", "border":"solid red"})
        ],style= {"height": "25vh", "border":"solid red"}),
            
        dbc.Row(children=[
            dbc.Col(children='g',
                    style= {"width": "50%", "border":"solid red"}),
            dbc.Col(children='h',
                    style= {"width": "50%", "border":"solid red"})
        ],style= {"height": "25vh", "border":"solid red"})
    ]
)

style={'border':'solid red'}

if __name__ == "__main__":
    app.run_server(debug=True, port = 8060, jupyter_mode = "external") 

Dash app running on http://127.0.0.1:8060/
